# Pipeline ETL Automatizado de Datos de Bicicletas Compartidas
## Introducción

Las empresas de movilidad generan grandes volúmenes de datos relacionados con viajes, tarifas, distancias y tiempos de transporte. Analizar estos datos permite comprender patrones de movilidad urbana y optimizar servicios.

En este proyecto se construye un pipeline ETL automatizado utilizando Python, Google Cloud y Apache Airflow para procesar datos de viajes de transporte urbano.

El pipeline realiza:


- Extracción de datos
- Limpieza
- Transformación
- Almacenamiento en la nube
- Carga en un Data Warehouse
- Automatización del flujo de datos

Esto replica la arquitectura utilizada en empresas de tecnología y movilidad para procesar datos operacionales.

## Problema del negocio

Las empresas de transporte necesitan responder preguntas como:

¿Cuáles son las horas con mayor demanda?

¿Qué zonas generan más ingresos?

¿Cómo cambia la demanda según el día?

Sin un sistema automatizado: el análisis requiere trabajo manual, los datos tardan en procesarse, se pierden oportunidades de optimización

Por eso es necesario implementar pipelines ETL automatizados.

## Objetivo general

Construir un pipeline ETL automatizado en la nube que procese datos de viajes de transporte urbano utilizando Python, Google Cloud Storage, BigQuery y Apache Airflow.

### Objetivos específicos

1. Extraer y limpiar datos de viajes utilizando Python y Pandas.

2. Transformar los datos para generar métricas de movilidad.

3. Cargar los datos procesados en Google Cloud Storage y BigQuery.

4. Automatizar el pipeline utilizando Apache Airflow.

## Dataset
NYC Taxi Trip Data
Link: https://github.com/DataTalksClub/nyc-tlc-data/releases

 ## Arquitectura del proyecto

 CSV Dataset
     │
     ▼
Python ETL
     │
     ▼
Limpieza de datos
     │
     ▼
Transformación
     │
     ▼
Google Cloud Storage
     │
     ▼
BigQuery
     │
     ▼
Análisis SQL
     │
     ▼
Automatización con Airflow

## Desarrollo del proyecto

### ETL en Python

In [1]:
##### scripts/etl_pipeline.py
# Cargar datos 
import pandas as pd

df = pd.read_parquet("/Users/carlosortiz/Downloads/fhvhv_tripdata_2026-01.parquet")

# Exploración inicial

print(df.head())



  hvfhs_license_num dispatching_base_num originating_base_num  \
0            HV0003               B03404               B03404   
1            HV0003               B03404               B03404   
2            HV0003               B03404               B03404   
3            HV0005               B03406                 None   
4            HV0005               B03406                 None   

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:08:19 2026-01-01 00:09:08 2026-01-01 00:09:37   
4 2026-01-01 00:21:16 2026-01-01 00:26:30 2026-01-01 00:26:40   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  ...  \
0 2026-01-01 01:13:23           262            79       4.300  ...   
1 2026-01-01 00:22:42           195            52       1.890  ...   
2 2026-0

In [2]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20940373 entries, 0 to 20940372
Data columns (total 25 columns):
 #   Column                Dtype         
---  ------                -----         
 0   hvfhs_license_num     object        
 1   dispatching_base_num  object        
 2   originating_base_num  object        
 3   request_datetime      datetime64[us]
 4   on_scene_datetime     datetime64[us]
 5   pickup_datetime       datetime64[us]
 6   dropoff_datetime      datetime64[us]
 7   PULocationID          int32         
 8   DOLocationID          int32         
 9   trip_miles            float64       
 10  trip_time             int64         
 11  base_passenger_fare   float64       
 12  tolls                 float64       
 13  bcf                   float64       
 14  sales_tax             float64       
 15  congestion_surcharge  float64       
 16  airport_fee           float64       
 17  tips                  float64       
 18  driver_pay            float64       
 19

In [ ]:
#Limpieza de datos

df = df.drop_duplicates()

df = df.dropna()

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

In [ ]:
# Crear nuevas variables

df["trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

df["trip_day"] = df["tpep_pickup_datetime"].dt.dayofweek

# Crear métricas

summary = df.groupby("pickup_hour").agg({

"trip_distance":"mean",
"total_amount":"mean",
"trip_duration":"mean"

}).reset_index()

summary.to_csv(
"data/processed/trip_summary.csv",
index=False
)

In [ ]:
#Subir datos a Google Cloud Storage

from google.cloud import storage

client = storage.Client()

bucket_name = "taxi-etl-bucket"

bucket = client.bucket(bucket_name)

blob = bucket.blob("processed/trip_summary.csv")

blob.upload_from_filename("data/processed/trip_summary.csv")

# Cargar datos a BigQuery

from google.cloud import bigquery

client = bigquery.Client()

table_id = "taxi-etl-project.taxi_dw.trip_summary"

job = client.load_table_from_uri(
"gs://taxi-etl-bucket/processed/trip_summary.csv",
table_id,
job_config=bigquery.LoadJobConfig(
source_format=bigquery.SourceFormat.CSV,
skip_leading_rows=1,
autodetect=True
)
)

job.result()

In [ ]:
# Consultas SQL

SELECT
pickup_hour,
AVG(trip_distance) as avg_distance,
AVG(total_amount) as avg_fare
FROM taxi_dw.trip_summary
GROUP BY pickup_hour
ORDER BY pickup_hour

In [ ]:
# Visualización

import matplotlib.pyplot as plt

plt.figure()

plt.plot(
summary["pickup_hour"],
summary["trip_distance"]
)

plt.title("Distancia promedio por hora")

plt.xlabel("Hora del día")

plt.ylabel("Distancia promedio")

plt.show()

In [ ]:
##### airflow/dag_taxi_pipeline.py

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import subprocess

def run_etl():
    subprocess.run(["python","scripts/etl_pipeline.py"])

with DAG(

dag_id="taxi_etl_pipeline",

start_date=datetime(2024,1,1),

schedule_interval="@daily",

catchup=False

) as dag:

    etl_task = PythonOperator(

    task_id="run_etl",

    python_callable=run_etl

)

etl_task

### Resultados
El pipeline permite:

- Procesar automáticamente datos de viajes

- Generar métricas de movilidad

- Analizar tendencias de transporte urbano

### Conclusiones

1. Se logró extraer y limpiar datos de movilidad.

2. Se generaron métricas útiles como duración promedio y distancia.

3. Los datos fueron almacenados en un Data Warehouse.

4. El pipeline fue automatizado con Airflow.

### Recomendaciones

Las empresas de transporte podrían: Optimizar precios dinámicos, Mejorar asignación de vehículos, Identificar horas pico